## Блок 1

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchaudio.transforms import MelSpectrogram, AmplitudeToDB, MFCC, Resample, FrequencyMasking, TimeMasking
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import pandas as pd
import librosa
from Levenshtein import distance as levenshtein_distance  # pip install python-Levenshtein
import os
import torchaudio
import torch.optim as optim
import random

# -----------------------------------------
# 1. Алфавит и кодирование
# -----------------------------------------
alphabet = {
    '<pad>': 0, ' ': 1, '#': 2,
    '0': 3, '1': 4, '2': 5, '3': 6, '4': 7, '5': 8, '6': 9, '7': 10, '8': 11, '9': 12,
    'А': 13, 'Б': 14, 'В': 15, 'Г': 16, 'Д': 17, 'Е': 18, 'Ж': 19, 'З': 20,
    'И': 21, 'Й': 22, 'К': 23, 'Л': 24, 'М': 25, 'Н': 26, 'О': 27, 'П': 28,
    'Р': 29, 'С': 30, 'Т': 31, 'У': 32, 'Ф': 33, 'Х': 34, 'Ц': 35, 'Ч': 36,
    'Ш': 37, 'Щ': 38, 'Ъ': 39, 'Ы': 40, 'Ь': 41, 'Э': 42, 'Ю': 43, 'Я': 44
}
num_classes = len(alphabet)
chars = list(alphabet.keys())


def encode_label(text: str, alpha: dict) -> torch.Tensor:
    return torch.tensor([alpha[ch] for ch in text if ch in alpha], dtype=torch.long)


# -----------------------------------------
# 2. AudioTransform: mel + MFCC + VAD + SpecAugment
# -----------------------------------------
class AudioTransform(nn.Module):
    def __init__(self,
                 orig_sr=8000,
                 target_sr=8000,
                 n_mels=128,
                 n_fft=1024,
                 hop_length=512,
                 n_mfcc=13):
        super().__init__()
        self.resampler = Resample(orig_freq=orig_sr, new_freq=target_sr)
        self.mel_spec = MelSpectrogram(sample_rate=target_sr,
                                       n_fft=n_fft,
                                       hop_length=hop_length,
                                       n_mels=n_mels)
        self.to_db = AmplitudeToDB()
        self.mfcc = MFCC(sample_rate=target_sr,
                         n_mfcc=n_mfcc,
                         melkwargs={'n_fft':n_fft,
                                    'n_mels':n_mels,
                                    'hop_length':hop_length})
        # SpecAugment transforms
        self.freq_mask = FrequencyMasking(freq_mask_param=15)
        self.time_mask = TimeMasking(time_mask_param=35)
        self.target_sr = target_sr

    def forward(self, waveform: torch.Tensor, augment: bool = False) -> torch.Tensor:
        # 1) Resample + VAD
        wav = self.resampler(waveform)
        wav_np = wav.squeeze(0).cpu().numpy()
        trimmed, _ = librosa.effects.trim(wav_np, top_db=20)
        wav = torch.from_numpy(trimmed).unsqueeze(0)
    
        # 2) Вычисляем mel и mfcc
        mel = self.mel_spec(wav)          # [1, n_mels, T]
        mel_db = self.to_db(mel)
        mfcc = self.mfcc(wav)             # [1, n_mfcc, T]
    
        # 3) Подгоняем высоту mfcc под mel_db
        diff = mel_db.size(1) - mfcc.size(1)
        if diff > 0:
            mfcc = F.pad(mfcc, (0,0, 0, diff))
        elif diff < 0:
            mfcc = mfcc[:, :mel_db.size(1), :]
    
        # 4) Склеиваем каналы
        features = torch.cat([mel_db, mfcc], dim=0)  # [2, n_mels, T]
    
        # 5) Sample‑level CMVN
        mu = features.mean()
        sigma = features.std()
        features = (features - mu) / (sigma + 1e-6)
    
        # 6) SpecAugment (если нужно)
        if augment:
            features = self.freq_mask(features)
            features = self.time_mask(features)
    
        return features


# -----------------------------------------
# 3. Dataset with caching
# -----------------------------------------
class CachedMorseDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform: AudioTransform,
                 alpha: dict, cache_dir='cache', augment=False):
        self.alpha = alpha
        self.augment = augment
        self.transform = transform
        os.makedirs(cache_dir, exist_ok=True)

        # 1) Первый проход: загрузка/вычисление фич и поиск max_T
        feats = []
        labels = []
        max_T = 0
        for _, row in df.iterrows():
            uid = row['id']
            cache_path = os.path.join(cache_dir, f"{uid}.pt")

            if os.path.exists(cache_path):
                feat = torch.load(cache_path)       # [2, F, T_i]
            else:
                wav, _ = torchaudio.load(f"morse_dataset/{uid}")
                feat = transform(wav, augment=False)
                torch.save(feat, cache_path)

            feats.append(feat)
            labels.append(encode_label(row['message'], alpha))
            max_T = max(max_T, feat.size(2))

        # 2) Второй проход: pad/trim всех фич до max_T по оси времени
        padded_feats = []
        for feat in feats:
            C, feat_freq, T = feat.shape
            if T < max_T:
                pad_amount = max_T - T
                # pad по последней (time) оси справа
                feat = F.pad(feat, (0, pad_amount, 0, 0, 0, 0))
            elif T > max_T:
                feat = feat[:, :, :max_T]
            padded_feats.append(feat)

        # 3) Собираем тензоры
        self.features = torch.stack(padded_feats)     # [N, 2, feat_freq, max_T]
        self.labels   = labels
        self.tgt_lens = torch.tensor([len(lbl) for lbl in labels], dtype=torch.long)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        feat = self.features[idx].clone()             # [2, feat_freq, max_T]
        if self.augment:
            # добавляем шум
            if random.random() < 0.5:
                feat = feat + torch.randn_like(feat) * 0.002
            # SpecAugment
            feat = self.transform.freq_mask(feat)
            feat = self.transform.time_mask(feat)

        label = self.labels[idx]
        in_len = feat.size(-1) // 4
        return feat, label, in_len, self.tgt_lens[idx]


# -----------------------------------------
# 4. Collate for CTC
# -----------------------------------------
def ctc_collate(batch):
    feats, labels, in_lens, tgt_lens = zip(*batch)
    feats = torch.stack(feats)
    labels_flat = torch.cat(labels)
    tgt_lens = torch.stack(tgt_lens)
    in_lens = torch.tensor(in_lens, dtype=torch.long)
    return feats, labels_flat, in_lens, tgt_lens


# -----------------------------------------
# 5. Model: CNN + BiLSTM + CTC
# -----------------------------------------
class CNNBiLSTMCTC(nn.Module):
    def __init__(self, num_classes=45, in_channels=2,
                 n_mels=128, lstm_hidden=384, lstm_layers=2, dropout=0.3):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, 1, 1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d((2, 1)),
            nn.Conv2d(64, 128, 3, 1, 1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d((2, 1)),
        )
        self.dropout = nn.Dropout(dropout)
        self.rnn_input = (n_mels // 4) * 128
        self.lstm = nn.LSTM(self.rnn_input, lstm_hidden,
                            num_layers=lstm_layers, bidirectional=True,
                            batch_first=True, dropout=dropout)
        self.classifier = nn.Linear(lstm_hidden * 2, num_classes)

    def forward(self, x):  # x: [B, C, F, T]
        y = self.cnn(x)  # [B,128,F/4,T]
        b, c, f, t = y.size()
        y = y.permute(0, 3, 1, 2).reshape(b, t, c * f)
        y = self.dropout(y)
        y, _ = self.lstm(y)
        y = self.classifier(y)
        return y.permute(1, 0, 2)  # [T, B, C]


# -----------------------------------------
# 6. Decoders: greedy + beam-search
# -----------------------------------------
def greedy_decode(logits):
    seqs = logits.argmax(-1).transpose(0, 1)
    results = []
    for s in seqs:
        prev = None
        chars_out = []
        for idx in s.cpu().tolist():
            if idx != prev and idx != 0:
                chars_out.append(chars[idx])
            prev = idx
        results.append("".join(chars_out))
    return results


def beam_search_decode(logits, beam_width=5):
    # logits: [T, B, C]
    T, B, C = logits.size()
    best = []
    logp = F.log_softmax(logits, dim=2).cpu().detach().numpy()
    for b in range(B):
        sequences = [([], 0.0)]
        for t in range(T):
            all_candidates = []
            for seq, score in sequences:
                for c in range(C):
                    all_candidates.append((seq + [c], score + logp[t, b, c]))
            # select best k
            sequences = sorted(all_candidates, key=lambda x: x[1], reverse=True)[:beam_width]
        # pick highest scoring
        seq, _ = sequences[0]
        # collapse
        pts = []
        prev = None
        for idx in seq:
            if idx != prev and idx != 0:
                pts.append(chars[idx])
            prev = idx
        best.append("".join(pts))
    return best


# -----------------------------------------
# 7. Training loop with OneCycleLR
# -----------------------------------------
def train_ctc(model, tr_ld, va_ld, epochs=20, lr=1e-3, wd=1e-5, device=None):
    dev = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(dev)
    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = optim.lr_scheduler.OneCycleLR(opt, max_lr=lr,
                                         steps_per_epoch=len(tr_ld), epochs=epochs)
    loss_fn = nn.CTCLoss(blank=0, zero_infinity=True)

    for ep in range(1, epochs + 1):
        # Training
        model.train()
        total_loss = 0.0
        for x, lbl, in_lens, tgt_lens in tqdm(tr_ld, desc=f"Train Ep{ep}"):
            x = x.to(dev)
            lbl = lbl.to(dev)
            in_lens = in_lens.to(dev)
            tgt_lens = tgt_lens.to(dev)
            opt.zero_grad()
            logits = model(x)
            loss = loss_fn(F.log_softmax(logits, 2), lbl, in_lens, tgt_lens)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()
            sched.step()
            total_loss += loss.item()
        avg_tr_loss = total_loss / len(tr_ld)

        # Validation
        model.eval()
        val_loss = 0.0
        dist = 0
        cnt = 0
        with torch.no_grad():
            for x, lbl, in_lens, tgt_lens in tqdm(va_ld, desc="Validation"):
                x = x.to(dev)
                lbl = lbl.to(dev)
                in_lens = in_lens.to(dev)
                tgt_lens = tgt_lens.to(dev)
                logits = model(x)
                val_loss += loss_fn(F.log_softmax(logits, 2), lbl, in_lens, tgt_lens).item()
                # decode
                preds = greedy_decode(logits)
                ptr = 0
                for i, l in enumerate(tgt_lens.cpu()):
                    truth = "".join(chars[idx] for idx in lbl[ptr:ptr + l].cpu().tolist())
                    ptr += l
                    dist += levenshtein_distance(preds[i], truth)
                    cnt += 1
        print(f"Ep{ep}: Train Loss={avg_tr_loss:.4f}, Val Loss={val_loss/len(va_ld):.4f}, CER={dist/cnt:.4f}")


# -----------------------------------------
# 8. Main
# -----------------------------------------
if __name__ == "__main__":
    df = pd.read_csv("train.csv")
    tr_df, va_df = train_test_split(df, test_size=0.2, random_state=42)

    # Base transform settings (orig_sr should match loaded audio)
    base_transform = AudioTransform(orig_sr=8000, target_sr=8000,
                                    n_mels=128, n_fft=1024,
                                    hop_length=512, n_mfcc=13)

    # Datasets & loaders
    tr_ds = CachedMorseDataset(tr_df, transform=base_transform,
                                alpha=alphabet, cache_dir='cache', augment=True)
    va_ds = CachedMorseDataset(va_df, transform=base_transform,
                                alpha=alphabet, cache_dir='cache', augment=False)

    tr_loader = DataLoader(tr_ds, batch_size=16, shuffle=True,
                           num_workers=0, collate_fn=ctc_collate)
    va_loader = DataLoader(va_ds, batch_size=8, shuffle=False,
                           num_workers=0, collate_fn=ctc_collate)

    # Model
    model = CNNBiLSTMCTC(num_classes=num_classes, in_channels=2,
                         n_mels=128, lstm_hidden=256,
                         lstm_layers=3, dropout=0.3)

    # Train
    train_ctc(model, tr_loader, va_loader, epochs=25, lr=1e-3)


In [ ]:
torch.save(model.state_dict(), "last_winner.pth")

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from torchaudio.transforms import Resample, MelSpectrogram, AmplitudeToDB, MFCC, FrequencyMasking, TimeMasking
import pandas as pd
import librosa
from tqdm import tqdm

# -----------------------------------------
# 1. Алфавит и индексы
# -----------------------------------------
alphabet = {
    '<pad>': 0, ' ': 1, '#': 2,
    '0': 3, '1': 4, '2': 5, '3': 6, '4': 7, '5': 8, '6': 9, '7': 10, '8': 11, '9': 12,
    'А': 13, 'Б': 14, 'В': 15, 'Г': 16, 'Д': 17, 'Е': 18, 'Ж': 19, 'З': 20,
    'И': 21, 'Й': 22, 'К': 23, 'Л': 24, 'М': 25, 'Н': 26, 'О': 27, 'П': 28,
    'Р': 29, 'С': 30, 'Т': 31, 'У': 32, 'Ф': 33, 'Х': 34, 'Ц': 35, 'Ч': 36,
    'Ш': 37, 'Щ': 38, 'Ъ': 39, 'Ы': 40, 'Ь': 41, 'Э': 42, 'Ю': 43, 'Я': 44
}
idx2char = {v: k for k, v in alphabet.items()}

# -----------------------------------------
# 2. AudioTransform: Resample + VAD + Mel + MFCC + CMVN
# -----------------------------------------
class AudioTransform(nn.Module):
    def __init__(self,
                 orig_sr=8000,
                 target_sr=8000,
                 n_mels=128,
                 n_fft=1024,
                 hop_length=512,
                 n_mfcc=13):
        super().__init__()
        self.resampler = Resample(orig_freq=orig_sr, new_freq=target_sr)
        self.mel_spec = MelSpectrogram(sample_rate=target_sr,
                                       n_fft=n_fft,
                                       hop_length=hop_length,
                                       n_mels=n_mels)
        self.to_db = AmplitudeToDB()
        self.mfcc = MFCC(sample_rate=target_sr,
                         n_mfcc=n_mfcc,
                         melkwargs={'n_fft': n_fft,
                                    'n_mels': n_mels,
                                    'hop_length': hop_length})
        # SpecAugment (unused in inference)
        self.freq_mask = FrequencyMasking(freq_mask_param=15)
        self.time_mask = TimeMasking(time_mask_param=35)

    def forward(self, waveform: torch.Tensor, augment: bool = False) -> torch.Tensor:
        # 1) Resample
        wav = self.resampler(waveform)              # [1, L]
        # 2) VAD trim
        wav_np = wav.squeeze(0).cpu().numpy()
        trimmed, _ = librosa.effects.trim(wav_np, top_db=20)
        wav = torch.from_numpy(trimmed).unsqueeze(0)  # [1, L']

        # 3) Mel -> dB
        mel = self.mel_spec(wav)                    # [1, n_mels, T]
        mel_db = self.to_db(mel)

        # 4) MFCC
        mfcc = self.mfcc(wav)                       # [1, n_mfcc, T]
        # pad/trim MFCC freq-dim to n_mels
        diff = mel_db.size(1) - mfcc.size(1)
        if diff > 0:
            mfcc = F.pad(mfcc, (0,0, 0,diff))
        elif diff < 0:
            mfcc = mfcc[:, :mel_db.size(1), :]

        # 5) Stack channels
        features = torch.cat([mel_db, mfcc], dim=0)  # [2, n_mels, T]

        # 6) Sample-level CMVN
        mu, sigma = features.mean(), features.std()
        features = (features - mu) / (sigma + 1e-6)

        return features  # [2, n_mels, T]

# -----------------------------------------
# 3. Модель: CNN + BiLSTM + CTC
# -----------------------------------------
class CNNBiLSTMCTC(nn.Module):
    def __init__(self, num_classes=45, in_channels=2,
                 n_mels=128, lstm_hidden=256, lstm_layers=3, dropout=0.3):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, 1, 1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d((2, 1)),
            nn.Conv2d(64, 128, 3, 1, 1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d((2, 1)),
        )
        self.dropout = nn.Dropout(dropout)
        self.rnn_input = (n_mels // 4) * 128
        self.lstm = nn.LSTM(self.rnn_input, lstm_hidden,
                            num_layers=lstm_layers, bidirectional=True,
                            batch_first=True, dropout=dropout)
        self.classifier = nn.Linear(lstm_hidden * 2, num_classes)

    def forward(self, x):
        # x: [B, 2, n_mels, T]
        y = self.cnn(x)                              # [B,128,n_mels/4,T]
        b, c, f, t = y.size()
        y = y.permute(0,3,1,2).reshape(b, t, c*f)     # [B,T,features]
        y = self.dropout(y)
        y, _ = self.lstm(y)                          # [B,T,2*hidden]
        y = self.classifier(y)                       # [B,T,num_classes]
        return y.permute(1,0,2)                      # [T,B,num_classes]

# -----------------------------------------
# 4. Greedy CTC Decode
# -----------------------------------------
def greedy_decode(logits: torch.Tensor, blank: int = 0):
    # logits: [T, B, C]
    pred = logits.argmax(dim=2).transpose(0,1)  # [B, T]
    results = []
    for seq in pred:
        seq = seq.cpu().tolist()
        prev = None
        chars = []
        for idx in seq:
            if idx != prev and idx != blank:
                chars.append(idx2char[idx])
            prev = idx
        results.append("".join(chars))
    return results  # list of strings, length B

# -----------------------------------------
# 5. Инференс
# -----------------------------------------
def infer(
    model_path: str = "last_winner.pth",
    test_csv:   str = "test.csv",
    audio_dir:  str = "morse_dataset",
    output_csv: str = "submission.csv",
    device:     str = None
):
    device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))

    # 5.1) Load model
    model = CNNBiLSTMCTC(
        num_classes=len(alphabet),
        in_channels=2,
        n_mels=128,
        lstm_hidden=256,
        lstm_layers=3,
        dropout=0.3
    )
    state = torch.load(model_path, map_location=device)
    model.load_state_dict(state)
    model.to(device)
    model.eval()

    # 5.2) Prepare transform
    transform = AudioTransform(orig_sr=8000, target_sr=8000,
                               n_mels=128, n_fft=1024,
                               hop_length=512, n_mfcc=13)

    # 5.3) Read test list
    df_test = pd.read_csv(test_csv)
    results = []

    for _, row in tqdm(df_test.iterrows(), total=len(df_test), desc="Inference"):
        uid = row['id']
        path = os.path.join(audio_dir, uid)
        # load audio
        wav, sr = torchaudio.load(path)         # [1, L], sr
        feat = transform(wav, augment=False)    # [2,128,T]
        inp = feat.unsqueeze(0).to(device)      # [1,2,128,T]

        with torch.no_grad():
            logits = model(inp)                 # [T,1,C]
            logp = F.log_softmax(logits, dim=2)

        pred = greedy_decode(logp)[0]           # string
        results.append({"id": uid, "message": pred})

    # 5.4) Save submission
    df_sub = pd.DataFrame(results)
    df_sub.to_csv(output_csv, index=False)
    print(f"✅ Saved submission: {output_csv}")

if __name__ == "__main__":
    infer(
        model_path="last_winner.pth",
        test_csv="test.csv",
        audio_dir="morse_dataset",
        output_csv="submission.csv"
    )